# Coffee Shop Sales Analysis

End-to-end analysis of coffee shop transaction data: cleaning, feature engineering, star-schema modeling, exploratory data analysis, and hypothesis testing.

**Contents**
1. Setup & Data Load
2. Initial Inspection
3. Data Cleaning
4. Feature Engineering
5. Star Schema Design & Export
6. Exploratory Data Analysis (EDA)
7. Hypothesis Testing
8. Supplementary Visualizations
9. Key Findings Summary

See `README.md` for a full project overview, dataset source, and how to reproduce this notebook.

## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Update this path to point to your local copy of the dataset
DATA_PATH = "data/Coffee Shop Sales.xlsx"

df = pd.read_excel(DATA_PATH)
df.head()

## 2. Initial Inspection

Basic shape, types, missing values, and duplicate check before any cleaning.

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

In [ ]:
df.duplicated().sum()

In [ ]:
df['product_category'].value_counts()

## 3. Data Cleaning

- Strip whitespace from all fields
- Convert columns to correct data types
- Parse date/time fields

In [ ]:
# Strip whitespace from string-like fields
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

In [ ]:
# Ensure correct dtypes
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['transaction_time'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S')
df['time_only'] = df['transaction_time'].dt.time

df['transaction_id'] = df['transaction_id'].astype(int)
df['transaction_qty'] = df['transaction_qty'].astype(int)
df['store_id'] = df['store_id'].astype(int)
df['product_id'] = df['product_id'].astype(int)
df['unit_price'] = df['unit_price'].astype(float)

df.dtypes

## 4. Feature Engineering

### 4.1 Total amount

In [ ]:
df['total_amount'] = df['unit_price'] * df['transaction_qty']

### 4.2 Date-based features

`month`, `week_number`, `day_name`, `is_weekend`.

In [ ]:
df['month'] = df['transaction_date'].dt.month_name().str[:3]
df['week_number'] = df['transaction_date'].dt.isocalendar().week
df['day_name'] = df['transaction_date'].dt.day_name()
df['is_weekend'] = df['day_name'].isin(['Saturday', 'Sunday'])

### 4.3 Holiday flag

Uses the `holidays` library to correctly identify US public holidays within the dataset's date range,
rather than a hand-maintained list — avoids missed/incorrect holiday dates.

```
pip install holidays
```

In [ ]:
import holidays
from datetime import date

us_holidays = holidays.US(years=df['transaction_date'].dt.year.unique().tolist())

start = df['transaction_date'].min().date()
end = df['transaction_date'].max().date()

relevant_holidays = {d: name for d, name in us_holidays.items() if start <= d <= end}

for d, name in sorted(relevant_holidays.items()):
    print(d, "-", name)

In [ ]:
df['is_holiday'] = df['transaction_date'].dt.date.isin(relevant_holidays.keys())
df['is_holiday'].value_counts()

### 4.4 Time-of-day bins

Bins `transaction_time` into Dawn/Early Morning, Morning, Afternoon, Evening, and Night windows.

In [ ]:
bins = [0, 4, 6, 12, 17, 21, 24]
labels = ['Night', 'Dawn/Early Morning', 'Morning', 'Afternoon', 'Evening', 'Late Night']

df['hour_bin'] = pd.cut(df['transaction_time'].dt.hour, bins=bins, labels=labels)
df['hour_bin'].value_counts()

### 4.5 Price tier

`unit_price` is heavily right-skewed (long tail of specialty items), so quartile-based cut points are used
instead of equal-width bins to keep each tier roughly balanced.

In [ ]:
df['unit_price'].quantile([0.25, 0.5, 0.75, 0.90, 0.95, 0.98, 0.99, 1])

In [ ]:
bins = [0, 2.50, 3.00, 3.75, df['unit_price'].max()]
labels = ['Low', 'Medium', 'High', 'Premium']

df['price_tier'] = pd.cut(df['unit_price'], bins=bins, labels=labels, include_lowest=True)
df['price_tier'].value_counts()

### 4.6 Size extraction

Extracts Small/Large/Regular from `product_detail` using a case-insensitive, word-boundary regex so it only
matches standalone "Sm"/"Lg"/"Rg" tokens (e.g. "Ethiopia Rg") rather than substrings inside other words.

Items that genuinely have no size variant (e.g. Bakery, Latte, Cappuccino, Espresso shot) are labeled
`'No Size Variant'` rather than left as an ambiguous "Unknown".

In [ ]:
import re

def extract_size(text):
    text = str(text)
    match = re.search(r'\b(sm|lg|rg)\b', text, flags=re.IGNORECASE)
    if match:
        size_map = {'sm': 'Small', 'lg': 'Large', 'rg': 'Regular'}
        return size_map[match.group(1).lower()]
    return 'No Size Variant'

df['size'] = df['product_detail'].apply(extract_size)
df['size'].value_counts()

In [ ]:
# Sanity check: confirm 'No Size Variant' rows are genuinely sizeless items,
# not a regex miss on a real drink category
df[df['size'] == 'No Size Variant']['product_category'].value_counts()

### 4.7 Export cleaned dataset

In [ ]:
df.to_csv("output/cleaned_data.csv", index=False)
df.head()

## 5. Star Schema Design & Export

**Grain:** one row = one product sold in one transaction (line-item level).

| Table | Key | Contents |
|---|---|---|
| `dim_date` | date_key | transaction_date, month, week_number, day_name, is_weekend, is_holiday |
| `dim_time` | time_key | time_only, hour_bin |
| `dim_store` | store_key | store_id, store_location |
| `dim_product` | product_key | product_id, product_category, product_type, product_detail, size, price_tier |
| `fact_sales` | transaction_id | date_key, time_key, store_key, product_key, transaction_qty, unit_price, total_amount |

In [ ]:
# Dim_Date
dim_date = df[['transaction_date', 'month', 'week_number', 'day_name',
               'is_weekend', 'is_holiday']].drop_duplicates().reset_index(drop=True)
dim_date['date_key'] = dim_date.index + 1
dim_date = dim_date[['date_key', 'transaction_date', 'month', 'week_number',
                      'day_name', 'is_weekend', 'is_holiday']]

In [ ]:
# Dim_Time
dim_time = df[['time_only', 'hour_bin']].drop_duplicates().reset_index(drop=True)
dim_time['time_key'] = dim_time.index + 1
dim_time = dim_time[['time_key', 'time_only', 'hour_bin']]

In [ ]:
# Dim_Store
dim_store = df[['store_id', 'store_location']].drop_duplicates().reset_index(drop=True)
dim_store['store_key'] = dim_store.index + 1
dim_store = dim_store[['store_key', 'store_id', 'store_location']]

In [ ]:
# Dim_Product
dim_product = df[['product_id', 'product_category', 'product_type',
                   'product_detail', 'size', 'price_tier']].drop_duplicates().reset_index(drop=True)
dim_product['product_key'] = dim_product.index + 1
dim_product = dim_product[['product_key', 'product_id', 'product_category',
                            'product_type', 'product_detail', 'size', 'price_tier']]

In [ ]:
# Fact_Sales
fact_sales = df.merge(dim_date[['date_key', 'transaction_date']], on='transaction_date', how='left')
fact_sales = fact_sales.merge(dim_time, on=['time_only', 'hour_bin'], how='left')
fact_sales = fact_sales.merge(dim_store, on=['store_id', 'store_location'], how='left')
fact_sales = fact_sales.merge(
    dim_product[['product_key', 'product_id', 'product_category', 'product_type',
                 'product_detail', 'size', 'price_tier']],
    on=['product_id', 'product_category', 'product_type', 'product_detail', 'size', 'price_tier'],
    how='left'
)

fact_sales = fact_sales[['transaction_id', 'date_key', 'time_key', 'store_key', 'product_key',
                          'transaction_qty', 'unit_price', 'total_amount']]

In [ ]:
# Validation: row counts should match, and no key columns should contain nulls
print("Original rows:", len(df), "| Fact rows:", len(fact_sales))
print(fact_sales[['date_key', 'time_key', 'store_key', 'product_key']].isnull().sum())

In [ ]:
dim_date.to_csv('output/dim_date.csv', index=False)
dim_time.to_csv('output/dim_time.csv', index=False)
dim_store.to_csv('output/dim_store.csv', index=False)
dim_product.to_csv('output/dim_product.csv', index=False)
fact_sales.to_csv('output/fact_sales.csv', index=False)

print("Star schema tables exported to /output")

## 6. Exploratory Data Analysis

### 6.1 Overview

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.describe())

### 6.2 Univariate — numeric distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df['unit_price'], bins=30, kde=True, ax=axes[0])
axes[0].set_title('Unit Price Distribution')

sns.histplot(df['transaction_qty'], bins=10, kde=True, ax=axes[1])
axes[1].set_title('Transaction Qty Distribution')

sns.histplot(df['total_amount'], bins=30, kde=True, ax=axes[2])
axes[2].set_title('Total Amount Distribution')

plt.tight_layout()
plt.show()

**Observation:** `unit_price` and `total_amount` are both heavily right-skewed — most transactions
are small, with a long tail of high-value outliers. This is why non-parametric tests are used in Section 7,
and why median is reported alongside mean throughout this analysis.

### 6.3 Univariate — categorical distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['product_category'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Transactions by Product Category')

df['size'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Transactions by Size')

df['store_location'].value_counts().plot(kind='bar', ax=axes[2])
axes[2].set_title('Transactions by Store Location')

plt.tight_layout()
plt.show()

In [ ]:
# Investigate large outliers in total_amount
df[df['total_amount'] > 100].sort_values('total_amount', ascending=False).head(20)

### 6.4 Bivariate — revenue drivers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('product_category')['total_amount'].sum().sort_values(ascending=False).plot(kind='bar', ax=axes[0, 0])
axes[0, 0].set_title('Total Revenue by Product Category')

df.groupby('hour_bin')['total_amount'].sum().plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Total Revenue by Hour Bin')

df.groupby('store_location')['total_amount'].sum().sort_values(ascending=False).plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Total Revenue by Store')

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df.groupby('day_name')['total_amount'].sum().reindex(day_order).plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Total Revenue by Day of Week')

plt.tight_layout()
plt.show()

## 7. Hypothesis Testing

`total_amount` is non-normally distributed (confirmed in Section 6.2), so **non-parametric tests** are used
throughout instead of t-tests/ANOVA: **Mann-Whitney U** for 2-group comparisons, **Kruskal-Wallis** for 3+
group comparisons, and **Chi-square** for associations between categorical variables.

Where a Kruskal-Wallis result is significant, a **Dunn's post-hoc test** (Bonferroni-corrected) is used to
identify which specific pair(s) differ.

### 7.1 Weekend vs weekday spending

**H0:** total_amount distribution is the same for weekend and weekday transactions.

In [ ]:
weekend_sales = df[df['is_weekend'] == True]['total_amount']
weekday_sales = df[df['is_weekend'] == False]['total_amount']

u_stat, p_value = stats.mannwhitneyu(weekend_sales, weekday_sales, alternative='two-sided')

print(f"U-statistic: {u_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print("Reject H0" if p_value < 0.05 else "Fail to reject H0")

In [ ]:
# Cross-check with Welch's t-test (not the primary test, given non-normal data —
# used only to confirm the Mann-Whitney result is robust to test choice)
t_stat, p_value = stats.ttest_ind(weekend_sales, weekday_sales, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

**Result:** No significant difference (p = 0.53 Mann-Whitney, p = 0.74 t-test). Spending behavior
is consistent across the week.

---

### 7.2 Store comparison

**H0:** total_amount distribution is the same across all three stores.

In [ ]:
hells_kitchen = df[df['store_location'] == "Hell's Kitchen"]['total_amount']
astoria = df[df['store_location'] == 'Astoria']['total_amount']
lower_manhattan = df[df['store_location'] == 'Lower Manhattan']['total_amount']

h_stat, p_value = stats.kruskal(hells_kitchen, astoria, lower_manhattan)

print(f"H-statistic: {h_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print("Reject H0" if p_value < 0.05 else "Fail to reject H0")

In [ ]:
# pip install scikit-posthocs
import scikit_posthocs as sp

dunn_result = sp.posthoc_dunn(df, val_col='total_amount', group_col='store_location', p_adjust='bonferroni')
dunn_result

In [ ]:
# Root-cause check: is the difference driven by price, quantity, or product mix?
df.groupby('store_location')[['total_amount', 'transaction_qty', 'unit_price']].describe()

In [ ]:
df.groupby('store_location')['product_category'].value_counts(normalize=True)

**Result:** Significant overall difference (p < 0.0001). Dunn's post-hoc shows Lower Manhattan differs
significantly from both Astoria and Hell's Kitchen (which don't differ from each other). Root cause: unit
pricing and product mix are nearly identical across stores, but Lower Manhattan customers buy more items
per transaction (avg 1.50 vs ~1.40 elsewhere) — a basket-size effect, not a pricing effect.

---

### 7.3 Holiday vs non-holiday spending

**H0:** total_amount distribution is the same for holiday and non-holiday transactions.

In [ ]:
holiday_sales = df[df['is_holiday'] == True]['total_amount']
non_holiday_sales = df[df['is_holiday'] == False]['total_amount']

u_stat, p_value = stats.mannwhitneyu(holiday_sales, non_holiday_sales, alternative='two-sided')

print(f"U-statistic: {u_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print("Reject H0" if p_value < 0.05 else "Fail to reject H0")

In [ ]:
df.groupby('is_holiday')['total_amount'].agg(['mean', 'median', 'count'])

**Result:** Statistically significant (p = 0.00023), but the effect size is trivial — median is
identical ($3.75) for both groups, and the mean differs by only ~$0.15 (~3%). With a large sample size,
even tiny differences become statistically detectable; this is not a practically meaningful revenue driver.

---

### 7.4 Time-of-day spending behavior

**H0:** total_amount distribution is the same across all `hour_bin` groups.

In [ ]:
morning = df[df['hour_bin'] == 'Morning']['total_amount']
afternoon = df[df['hour_bin'] == 'Afternoon']['total_amount']
evening = df[df['hour_bin'] == 'Evening']['total_amount']
dawn = df[df['hour_bin'] == 'Dawn/Early Morning']['total_amount']

h_stat, p_value = stats.kruskal(morning, afternoon, evening, dawn)

print(f"H-statistic: {h_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print("Reject H0" if p_value < 0.05 else "Fail to reject H0")

In [ ]:
df['hour_bin'].value_counts()

**Result:** No significant difference in per-transaction spending across time-of-day groups (p = 0.46),
despite Morning generating far more *total* revenue. This is a volume story, not a spending-behavior story:
Morning simply has far more transactions (85,865) than Afternoon (44,464), Evening (14,193), or
Dawn/Early Morning (4,594) — the typical transaction size is statistically the same regardless of when it happens.

---

### 7.5 Product category × size association

**H0:** `product_category` and `size` are independent.

In [ ]:
contingency_table = pd.crosstab(df['product_category'], df['size'])
print(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"\nChi2 statistic: {chi2:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")
print("Reject H0" if p_value < 0.05 else "Fail to reject H0")

**Result:** Significant association (p < 0.0001), but it reflects menu structure rather than customer
behavior — only Coffee, Tea, and Drinking Chocolate have real size variants; other categories are
inherently sizeless.

---

### 7.6 Product category × store location association

**H0:** `product_category` and `store_location` are independent.

In [ ]:
contingency_table2 = pd.crosstab(df['product_category'], df['store_location'])
print(contingency_table2)

chi2, p_value, dof, expected = chi2_contingency(contingency_table2)

print(f"\nChi2 statistic: {chi2:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")
print("Reject H0" if p_value < 0.05 else "Fail to reject H0")

**Result:** Statistically significant (p < 0.0001), but proportionally modest — category rank order
(Coffee > Tea > Bakery > ...) is consistent across all three stores, with Lower Manhattan showing only a
slight lean toward Bakery/Flavours, consistent with its larger basket-size finding above.

## 8. Supplementary Visualizations

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='store_location', y='total_amount', showfliers=False)
plt.title('Total Amount Distribution by Store (outliers hidden for clarity)')
plt.ylabel('Total Amount ($)')
plt.xlabel('Store Location')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(data=df, x='store_location', y='total_amount', cut=0)
plt.title('Distribution Shape of Total Amount by Store')
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='is_weekend', y='total_amount', showfliers=False)
plt.xticks([0, 1], ['Weekday', 'Weekend'])
plt.title('Total Amount: Weekday vs Weekend (p = 0.53, not significant)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='hour_bin', y='total_amount',
            order=['Dawn/Early Morning', 'Morning', 'Afternoon', 'Evening'], errorbar='ci')
plt.title('Avg Total Amount by Time of Day (error bars = 95% CI)')
plt.ylabel('Average Total Amount ($)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='hour_bin', order=['Dawn/Early Morning', 'Morning', 'Afternoon', 'Evening'])
plt.title('Transaction Count by Time of Day (Morning drives volume, not spend)')
plt.ylabel('Number of Transactions')
plt.show()

In [ ]:
pivot = pd.crosstab(df['product_category'], df['store_location'], normalize='columns') * 100

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrBr', cbar_kws={'label': '% of store transactions'})
plt.title('Product Category Mix by Store (%)')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='store_location', y='transaction_qty', showfliers=False)
plt.title('Transaction Quantity by Store (Lower Manhattan has larger baskets)')
plt.ylabel('Items per Transaction')
plt.show()

In [ ]:
numeric_cols = ['unit_price', 'transaction_qty', 'total_amount']
plt.figure(figsize=(5, 4))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Between Numeric Variables')
plt.show()

## 9. Key Findings Summary

| Finding | Statistically significant? | Practically meaningful? |
|---|---|---|
| Weekend vs weekday spending | No | — |
| Store distributions differ | Yes | **Yes** — Lower Manhattan has larger basket sizes |
| Holiday vs non-holiday spending | Yes | No — ~$0.15 difference, same median |
| Time-of-day spending behavior | No | — (but transaction *volume* varies hugely) |
| Category × size association | Yes | No — reflects menu structure |
| Category × store association | Yes | Modest |

**Actionable takeaways:**
- **Lower Manhattan** generates comparable revenue to the other stores with fewer transactions, driven by
  larger basket sizes rather than pricing or product mix — worth investigating as a replicable pattern.
- **Morning** drives the most revenue purely through transaction volume, not higher per-visit spending —
  growth in slower periods should target foot traffic, not upselling.
- Weekday/weekend and holiday effects are statistically real in places but too small to inform strategy.

Full business write-up with recommendations: see `docs/Coffee_Shop_Sales_Insights_Report.docx` (or the
project README) for the stakeholder-facing version of these findings.